In [ ]:
import sqlite3
import datetime

def get_db_connection():
    conn = sqlite3.connect('healthcare.db')
    conn.row_factory = sqlite3.Row
    return conn

# 복용 데이터

```
CREATE TABLE IF NOT EXISTS medications (
    cond_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    medicine VARCHAR(255) NOT NULL,
    recorded_at DATETIME NOT NULL,  -- 기록 날짜
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
```

## 복용 데이터 추가

In [ ]:
def save_medication(user_id, medicine):
    conn = get_db_connection()
    cursor = conn.cursor()

    recorded_at = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    try:
        cursor.execute(
            "INSERT INTO medications (user_id, medicine, recorded_at) VALUES (?, ?, ?)",
            (user_id, medicine, recorded_at)
        )
        conn.commit()
        print(f"약복용 데이터 저장 완료 '{medicine}'. 사용자: {user_id}.")
    except sqlite3.Error as e:
        print(f"약 복용 데이터 저장 오류: {e}")
        conn.rollback()
    finally:
        conn.close()

## 복용 데이터 조회

In [ ]:
def get_medication(user_id):
    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute(
            "SELECT * FROM medications WHERE user_id = ? ORDER BY recorded_at DESC",
            (user_id,)
        )
        medications = cursor.fetchall()
        if medications:
            print(f"사용자 {user_id}의 복용 데이터 불러오기 완료.")
            return medications
        else:
            print(f"사용자 {user_id}에 대한 복용 데이터 없음.")
            return None
    except sqlite3.Error as e:
        print(f"복용 데이터 불러오기 오류: {e}")
        return None
    finally:
        conn.close()

# 컨디션 데이터

```
CREATE TABLE IF NOT EXISTS conditions (
    cond_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    meal VARCHAR(255) NOT NULL,
    sleep VARCHAR(255) NOT NULL,
    mood VARCHAR(255) NOT NULL,
    pain VARCHAR(255) NOT NULL,
    recorded_at DATETIME NOT NULL,  -- 기록 날짜
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
```

## 컨디션 데이터 추가

In [ ]:
def save_condition(user_id, meal, sleep, mood, pain):
    conn = get_db_connection()
    cursor = conn.cursor()

    recorded_at = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    try:
        cursor.execute(
            """INSERT INTO conditions
            (user_id, meal, sleep, mood, pain, recorded_at)
            VALUES (?, ?, ?, ?, ?, ?)""",
            (user_id, meal, sleep, mood, pain, recorded_at)
        )
        conn.commit()
        print(f"컨디션 데이터 저장 완료. 사용자 아이디: {user_id}.")
    except sqlite3.Error as e:
        print(f"컨디션 데이터 저장 오류: {e}")
        conn.rollback()
    finally:
        conn.close()

## 사용자별 최근 컨디션 데이터 조회

In [ ]:
def get_latest_condition(user_id):
    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute(
            "SELECT * FROM conditions WHERE user_id = ? ORDER BY recorded_at DESC LIMIT 1",
            (user_id,)
        )
        condition = cursor.fetchone()
        if condition:
            print(f"사용자 {user_id} 최신 컨디션 데이터 불러오기 완료.")
            return dict(condition) # Convert row object to dictionary
        else:
            print(f"사용자 {user_id}에 대한 컨디션 데이터 없음.")
            return None
    except sqlite3.Error as e:
        print(f"최신 컨디션 데이터 불러오기 오류: {e}")
        return None
    finally:
        conn.close()

# alerts 데이터

```
CREATE TABLE IF NOT EXISTS alerts (
    alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER ,  -- Foreign Key,  -- 어르신(대상자)
    cause TEXT NOT NULL,  -- 이상신호 근거
    level VARCHAR(10) NOT NULL CHECK(level IN ('낮음', '중간', '높음')),
    status VARCHAR(10) NOT NULL CHECK(status IN ('대기', '확인')),
    created_at date NOT NULL,  -- 이상신호 발생 시각
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);
```

## alert 함수에 데이터 추가

### 대화 내용 분석 함수

In [ ]:
from openai import OpenAI
from google.colab import userdata
import json

# OpenAI API 키 불러오기
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

def chat_analysis(user_id, chat):
    system_prompt = """
    너는 사용자의 대화 내용을 분석하는 챗봇이야.
    사용자가 말하는 내용에서 무엇 때문에 불편해 하는지, 그리고 그 정도가 어느 수준인지 파악해야 해.
    분석 결과는 JSON 형식으로만 응답해야 하며, 다음 형식을 따라야 해:
    {
        "cause": "불편함의 원인을 한 문장으로 요약",
        "level": "불편함의 수준 (낮음, 중간, 높음 중 하나)"
    }
    불편함이 없는 일반적인 대화인 경우, cause는 '특이사항 없음'으로, level은 '낮음'으로 설정해.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            response_format={ "type": "json_object" },
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": chat}
            ]
        )
        result = response.choices[0].message.content

        parsed_result = json.loads(result)
        cause = parsed_result.get('cause', '분석 불가')
        level = parsed_result.get('level', '낮음')
        print(f"사용자 {user_id}의 대화 분석 완료: 원인 - {cause}, 수준 - {level}")
        return cause, level
    except Exception as e:
        print(f"대화 분석 중 오류 발생: {e}")
        return "분석 오류", "낮음"

In [ ]:
# 테스트

test_user_id = 1

sample1 = "어제부터 무릎이 아직도 시큰그리하다."
cause, level = chat_analysis(test_user_id, sample1)
print(f"Sample 1 분석 결과: 원인 - {cause}, 수준 - {level}")

sample2 = "점심은 아까 국에 밥 말아 뭇다."
cause_s2, level_s2 = chat_analysis(test_user_id, sample2)
print(f"Sample 2 분석 결과: 원인 - {cause_s2}, 수준 - {level_s2}")

사용자 1의 대화 분석 완료: 원인 - 무릎 통증으로 인해 불편함을 느끼고 있음, 수준 - 중간
Sample 1 분석 결과: 원인 - 무릎 통증으로 인해 불편함을 느끼고 있음, 수준 - 중간
사용자 1의 대화 분석 완료: 원인 - 특이사항 없음, 수준 - 낮음
Sample 2 분석 결과: 원인 - 특이사항 없음, 수준 - 낮음


### 데이터 저장

In [ ]:
def save_alert(user_id, cause, level, status):
    conn = get_db_connection()
    cursor = conn.cursor()
    created_at = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    try:
        cursor.execute(
            "INSERT INTO alerts (user_id, cause, level, status, created_at) VALUES (?, ?, ?, ?, ?)",
            (user_id, cause, level, status, created_at)
        )
        conn.commit()
        print(f"사용자 {user_id}의 위험 신호 데이터 추가. Cause: {cause}, Level: {level}, Status: {status}")
    except sqlite3.Error as e:
        print(f"위험 신호 데이터 저장 오류: {e}")
        conn.rollback()
    finally:
        conn.close()

## 사용자별 alerts 데이터 조회

In [ ]:
def get_alert(user_id):
    conn = get_db_connection()
    cursor = conn.cursor()

    try:
        cursor.execute(
            "SELECT * FROM alerts WHERE user_id = ? ORDER BY created_at DESC",
            (user_id,)
        )
        alerts = cursor.fetchall()
        if alerts:
            print(f"사용자 {user_id}의 위험 신호 데이터 불러오기 완료.")
            return alerts
        else:
            print(f"사용자 {user_id}에 대한 위험 신호 데이터 없음.")
            return None
    except sqlite3.Error as e:
        print(f"위험 신호 데이터 불러오기 오류: {e}")
        return None
    finally:
        conn.close()